# COVID-19 Clinical Data Lakehouse — Data Architecture Documentation

**Catalog:** `covid19_lakehouse` | **Schema:** `gold` | **Dataset:** Synthea 100k Patients (Synthetic Clinical Data)

**Architecture:** Medallion (Bronze → Silver → Gold) using Lakeflow Spark Declarative Pipelines (SDP) + Auto Loader

---

This document provides the complete **Logical Data Model** and **Physical Data Model** for the Gold layer of the COVID-19 Clinical Data Lakehouse, implemented as a Kimball-style Star Schema on Databricks Delta Lake.

**Document Sections:**
1. **Part 1 — Logical Data Model**: ERD breakdown, entity catalog, cardinality & relationships, business definitions and grain
2. **Part 2 — Physical Data Model**: Data dictionary (columns, types, PKs, FKs), Delta Lake physical configurations

## Part 1 — Logical Data Model

### 1.1 Entity Overview

The Gold layer implements a **Kimball Star Schema** with 18 core entities across 4 functional groups:

| # | Entity | Type | Role | Rows | Grain / Description |
| --- | --- | --- | --- | --- | --- |
| 1 | `dim_patient` | MATERIALIZED\_VIEW | Dimension (SCD2) | 124,150 | One row per patient version (SCD Type 2) |
| 2 | `dim_date` | MATERIALIZED\_VIEW | Dimension (Calendar) | 40,908 | One row per calendar date (1909–2020) |
| 3 | `dim_condition` | STREAMING\_TABLE | Dimension | 189 | One row per unique medical condition (SNOMED-CT) |
| 4 | `dim_encounter_type` | STREAMING\_TABLE | Dimension | 48 | One row per encounter classification code |
| 5 | `dim_medication` | STREAMING\_TABLE | Dimension | 181 | One row per unique medication (RxNorm) |
| 6 | `dim_observation_type` | STREAMING\_TABLE | Dimension | 215 | One row per observation/measurement type |
| 7 | `dim_organization` | STREAMING\_TABLE | Dimension | 9,175 | One row per healthcare facility/hospital |
| 8 | `dim_payer` | STREAMING\_TABLE | Dimension | 10 | One row per insurance provider |
| 9 | `dim_procedure` | STREAMING\_TABLE | Dimension | 177 | One row per unique medical procedure (SNOMED-CT) |
| 10 | `dim_provider` | STREAMING\_TABLE | Dimension | 60,534 | One row per healthcare provider/physician |
| 11 | `dim_reason` | MATERIALIZED\_VIEW | Dimension (Conformed) | 123 | One row per medical reason + Unknown Member (-1) |
| 12 | `fact_encounters` | STREAMING\_TABLE | Fact | 3,183,531 | One row per healthcare encounter |
| 13 | `fact_observations` | STREAMING\_TABLE | Fact | 16,219,969 | One row per clinical observation |
| 14 | `fact_conditions` | STREAMING\_TABLE | Fact (Factless) | 1,143,900 | One row per condition diagnosis occurrence |
| 15 | `fact_medications` | STREAMING\_TABLE | Fact | 4,226,915 | One row per medication prescription |
| 16 | `fact_procedures` | STREAMING\_TABLE | Fact | 979,564 | One row per medical procedure performed |
| 17 | `fact_payer_coverage_period` | STREAMING\_TABLE | Fact | 409,553 | One row per patient–payer coverage period |
| 18 | `fact_dq_quarantine_log` | STREAMING\_TABLE | Fact (Governance) | 0 | One row per quarantine event per source table per batch |

**Supporting Objects (not part of the core 18):**

| # | Object | Type | Description |
| --- | --- | --- | --- |
| 19 | `dim_patient_scd` | STREAMING\_TABLE | Internal SCD2 backing table for `dim_patient` via `dlt.apply_changes` |
| 20 | `star_schema_metadata` | MANAGED | PK/FK constraint catalog documenting all key relationships |
| 21–25 | `*_metric_view` (×5) | METRIC\_VIEW | Semantic layer KPI views for dashboards (encounters, conditions, medications, observations, procedures) |

### 1.2 Dimension Entities (11 Tables)

#### 1.2.1 dim\_patient — Patient Dimension (SCD Type 2)

| Property | Value |
| --- | --- |
| **Primary Key** | `patient_key` (BIGINT — `xxhash64(patient_id)`) |
| **Business Key** | `patient_id` (STRING — Synthea source ID) |
| **Grain** | One row per patient **version** (SCD2 — multiple rows per patient when attributes change) |
| **SCD Type** | Type 2 via `dlt.apply_changes` |
| **SCD2 Columns** | `__START_AT` (version start timestamp), `__END_AT` (version end timestamp, NULL = current), `is_current` (boolean — derived from `__END_AT IS NULL`) |
| **Row Count** | 124,150 |
| **Description** | Stores patient demographics (name, birth/death dates, gender, race, ethnicity, marital status, address, geo-coordinates). The SCD2 design preserves historical patient profiles for point-in-time accurate analysis. |

#### 1.2.2 dim\_date — Calendar Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `date_key` (INT — YYYYMMDD numeric format) |
| **Grain** | One row per calendar date (1909-01-01 to 2020-12-31) |
| **Row Count** | 40,908 |
| **Description** | Conformed calendar dimension with day, month, quarter, year, week-of-year, day name, month name, and weekend flag. Used by all fact tables for date-based joins. |

#### 1.2.3 dim\_condition — Condition Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `condition_key` (BIGINT — `xxhash64(condition_code)`) |
| **Business Key** | `condition_code` (STRING — SNOMED-CT) |
| **Grain** | One row per unique medical condition |
| **Row Count** | 189 |
| **Description** | Standardized medical diagnosis lookup (SNOMED-CT codes + descriptive names). Referenced by `fact_conditions`. |

#### 1.2.4 dim\_encounter\_type — Encounter Type Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `encounter_type_key` (BIGINT — `xxhash64(encounter_code)`) |
| **Business Key** | `encounter_code` (STRING) |
| **Grain** | One row per encounter classification |
| **Row Count** | 48 |
| **Description** | Visit class lookup (ambulatory, emergency, inpatient, wellness, urgent care). Referenced by `fact_encounters`. |

#### 1.2.5 dim\_medication — Medication Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `medication_key` (BIGINT — `xxhash64(medication_code)`) |
| **Business Key** | `medication_code` (STRING — RxNorm) |
| **Grain** | One row per unique medication |
| **Row Count** | 181 |
| **Description** | Medication lookup (brand/generic names + standard codes). Referenced by `fact_medications`. |

#### 1.2.6 dim\_observation\_type — Observation Type Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `observation_type_key` (BIGINT — `xxhash64(observation_code)`) |
| **Business Key** | `observation_code` (STRING) |
| **Grain** | One row per observation/measurement type |
| **Row Count** | 215 |
| **Description** | Clinical measurement type lookup (e.g., Body Height, Body Weight, Blood Pressure) with units. Referenced by `fact_observations`. |

#### 1.2.7 dim\_organization — Organization Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `organization_key` (BIGINT — `xxhash64(organization_id)`) |
| **Business Key** | `organization_id` (STRING — Synthea source ID) |
| **Grain** | One row per healthcare facility/hospital |
| **Row Count** | 9,175 |
| **Description** | Facility lookup with name, address, geo-coordinates, phone, revenue, and utilization. Referenced by `fact_encounters` and `dim_provider` (snowflake). |

#### 1.2.8 dim\_payer — Payer Dimension (Conformed)

| Property | Value |
| --- | --- |
| **Primary Key** | `payer_key` (BIGINT — `xxhash64(payer_id)`) |
| **Business Key** | `payer_id` (STRING — Synthea source ID) |
| **Grain** | One row per insurance provider |
| **Row Count** | 10 |
| **Description** | Conformed insurance payer dimension. Referenced by `fact_encounters`, `fact_medications`, and `fact_payer_coverage_period`. |

#### 1.2.9 dim\_procedure — Procedure Dimension

| Property | Value |
| --- | --- |
| **Primary Key** | `procedure_key` (BIGINT — `xxhash64(procedure_code)`) |
| **Business Key** | `procedure_code` (STRING — SNOMED-CT) |
| **Grain** | One row per unique medical procedure |
| **Row Count** | 177 |
| **Description** | Standardized medical procedure lookup. Referenced by `fact_procedures`. |

#### 1.2.10 dim\_provider — Provider Dimension (Snowflake)

| Property | Value |
| --- | --- |
| **Primary Key** | `provider_key` (BIGINT — `xxhash64(provider_id)`) |
| **Business Key** | `provider_id` (STRING — Synthea source ID) |
| **Foreign Key** | `organization_key` → `dim_organization.organization_key` |
| **Grain** | One row per healthcare provider/physician |
| **Row Count** | 60,534 |
| **Description** | Physician/provider lookup with name, gender, specialty, and organization assignment. Forms a snowflake relationship with `dim_organization`. |

#### 1.2.11 dim\_reason — Reason Dimension (Conformed)

| Property | Value |
| --- | --- |
| **Primary Key** | `reason_key` (BIGINT — `xxhash64(reason_code)`) |
| **Business Key** | `reason_code` (STRING — SNOMED-CT) |
| **Grain** | One row per unique medical reason + 1 Unknown Member row (`reason_key = -1`) |
| **Row Count** | 123 |
| **Description** | Conformed reason dimension unified from encounters, medications, and procedures. The Unknown Member row (-1) handles NULL reasons, common for routine/wellness visits (~78% of encounters). Referenced by `fact_encounters`, `fact_medications`, and `fact_procedures`. |

### 1.3 Fact Entities (6 Transaction Facts + 1 Governance Fact)

#### 1.3.1 fact\_encounters — Encounter Fact (Hub Table)

| Property | Value |
| --- | --- |
| **Primary Key** | `encounter_id` (STRING — UUID Business Key, also a Degenerate Dimension) |
| **Grain** | One row per healthcare encounter |
| **Row Count** | 3,183,531 |
| **Foreign Keys** | `patient_key` → dim\_patient, `provider_key` → dim\_provider, `organization_key` → dim\_organization, `payer_key` → dim\_payer, `encounter_type_key` → dim\_encounter\_type, `reason_key` → dim\_reason, `start_date_key` → dim\_date, `stop_date_key` → dim\_date |
| **Degenerate Dimension** | `encounter_id` is referenced by fact\_observations, fact\_conditions, fact\_medications, and fact\_procedures as a direct join key |
| **Measures** | `duration_minutes`, `base_encounter_cost`, `total_claim_cost`, `payer_coverage`, `patient_out_of_pocket`, `is_covid_related` |
| **Description** | Central fact table of the star schema — records every patient visit with cost breakdowns, timing, and COVID-19 flags. Acts as a hub for other fact tables via the `encounter_id` degenerate dimension. |

#### 1.3.2 fact\_observations — Observation Fact

| Property | Value |
| --- | --- |
| **Primary Key** | `observation_key` (BIGINT — `xxhash64` of patient + encounter + code + timestamp) |
| **Grain** | One row per clinical observation/measurement |
| **Row Count** | 16,219,969 (largest table) |
| **Foreign Keys** | `patient_key` → dim\_patient, `observation_type_key` → dim\_observation\_type, `date_key` → dim\_date |
| **Degenerate Dimension** | `encounter_id` → fact\_encounters (nullable — 2.69% of observations are independent of a specific encounter) |
| **Measures** | `value_numeric` (DOUBLE), `value_text` (STRING) — mutually exclusive; one is NULL when the other is populated |
| **Description** | Clinical observations and lab results (vital signs, body measurements, COVID tests). Numeric values support trend analysis; text values support categorical test results (Positive/Negative). |

#### 1.3.3 fact\_conditions — Condition Fact (Factless Event / Bridge)

| Property | Value |
| --- | --- |
| **Primary Key** | `condition_fact_key` (BIGINT — `xxhash64` of patient + encounter + condition + start date) |
| **Grain** | One row per condition diagnosis occurrence (same condition can appear multiple times for the same patient) |
| **Row Count** | 1,143,900 |
| **Foreign Keys** | `patient_key` → dim\_patient, `condition_key` → dim\_condition, `start_date_key` → dim\_date, `stop_date_key` → dim\_date |
| **Degenerate Dimension** | `encounter_id` → fact\_encounters |
| **Bridge Role** | The combination of `patient_key` + `condition_key` enables comorbidity analysis (identifying patients with multiple concurrent conditions) |
| **Measures** | `condition_duration_days`, `is_active` (boolean — derived from `stop_date_key IS NULL`) |
| **Description** | Factless event table recording each condition diagnosis. Serves a dual purpose as a bridge table for comorbidity analysis. |

#### 1.3.4 fact\_medications — Medication Fact

| Property | Value |
| --- | --- |
| **Primary Key** | `medication_fact_key` (BIGINT — `xxhash64` of patient + encounter + medication + start date) |
| **Grain** | One row per medication prescription record |
| **Row Count** | 4,226,915 |
| **Foreign Keys** | `patient_key` → dim\_patient, `payer_key` → dim\_payer, `medication_key` → dim\_medication, `reason_key` → dim\_reason, `start_date_key` → dim\_date, `stop_date_key` → dim\_date |
| **Degenerate Dimension** | `encounter_id` → fact\_encounters |
| **Measures** | `dispenses`, `base_cost`, `total_cost`, `payer_coverage`, `patient_out_of_pocket`, `treatment_duration_days`, `is_active` |
| **Description** | Medication prescriptions with full cost breakdown (base, total, payer coverage, out-of-pocket) and treatment duration tracking. Active prescriptions have NULL `stop_date_key` and `treatment_duration_days` to prevent shifting daily metrics. |

#### 1.3.5 fact\_procedures — Procedure Fact

| Property | Value |
| --- | --- |
| **Primary Key** | `procedure_fact_key` (BIGINT — `xxhash64` of patient + encounter + procedure + date) |
| **Grain** | One row per medical procedure performed |
| **Row Count** | 979,564 |
| **Foreign Keys** | `patient_key` → dim\_patient, `procedure_key` → dim\_procedure, `reason_key` → dim\_reason, `date_key` → dim\_date |
| **Degenerate Dimension** | `encounter_id` → fact\_encounters |
| **Measures** | `base_cost` (DECIMAL — note: source data does not include separate payer coverage breakdown) |
| **Description** | Medical procedures performed during encounters. Cost tracking is limited to `base_cost` as the source (procedures.csv) does not provide insurance breakdown details. |

#### 1.3.6 fact\_payer\_coverage\_period — Payer Coverage Period Fact

| Property | Value |
| --- | --- |
| **Primary Key** | `coverage_fact_key` (BIGINT — `xxhash64` of patient + payer + start\_year) |
| **Grain** | One row per patient–payer coverage period |
| **Row Count** | 409,553 |
| **Foreign Keys** | `patient_key` → dim\_patient, `payer_key` → dim\_payer |
| **Measures** | `start_year`, `end_year`, `duration_years` (calculated: `end_year - start_year`) |
| **Attributes** | `ownership` (Self/Spouse/Guardian — specific to the coverage period, not the payer) |
| **Description** | Insurance coverage periods per patient. Tracks when a patient was covered by a specific payer, including the ownership relationship type. Unlike other facts, uses year-level granularity (no date\_key FKs). |

#### 1.3.7 fact\_dq\_quarantine\_log — Data Quality Governance Fact

| Property | Value |
| --- | --- |
| **Primary Key** | `quarantine_key` (BIGINT) |
| **Grain** | One row per quarantine event (per source table, per rule, per batch) |
| **Row Count** | 0 (accumulates on pipeline refresh with rejected records) |
| **Foreign Keys** | `batch_date_key` → dim\_date |
| **Measures** | `rejected_row_count` |
| **Attributes** | `source_table` (which silver table the rejection occurred in), `rule_violated` (DQ rule description), `logged_at` (timestamp) |
| **Description** | Governance fact table logging data quality rejections from the Silver layer. Each row captures the source table, violated rule, and count of rejected records per streaming batch. Provides an auditable trail of DQ enforcement across the pipeline. |

### 1.4 Cardinality & Relationships (ERD)

The star schema centers on `fact_encounters` as the hub fact table. All other fact tables connect to it via the `encounter_id` degenerate dimension. Dimensions are shared (conformed) across fact tables where applicable.

#### Dimension-to-Fact Relationships (1:N)

| Dimension (Parent) | Fact (Child) | FK Column | Cardinality | Notes |
| --- | --- | --- | --- | --- |
| `dim_patient` | `fact_encounters` | `patient_key` | 1:N | One patient → many encounters |
| `dim_patient` | `fact_observations` | `patient_key` | 1:N | One patient → many observations |
| `dim_patient` | `fact_conditions` | `patient_key` | 1:N | One patient → many condition diagnoses |
| `dim_patient` | `fact_medications` | `patient_key` | 1:N | One patient → many prescriptions |
| `dim_patient` | `fact_procedures` | `patient_key` | 1:N | One patient → many procedures |
| `dim_patient` | `fact_payer_coverage_period` | `patient_key` | 1:N | One patient → many coverage periods |
| `dim_provider` | `fact_encounters` | `provider_key` | 1:N | One provider → many encounters |
| `dim_organization` | `fact_encounters` | `organization_key` | 1:N | One facility → many encounters |
| `dim_organization` | `dim_provider` | `organization_key` | 1:N | **Snowflake**: One facility → many providers |
| `dim_payer` | `fact_encounters` | `payer_key` | 1:N | Conformed — shared across 3 facts |
| `dim_payer` | `fact_medications` | `payer_key` | 1:N | Conformed — shared across 3 facts |
| `dim_payer` | `fact_payer_coverage_period` | `payer_key` | 1:N | Conformed — shared across 3 facts |
| `dim_encounter_type` | `fact_encounters` | `encounter_type_key` | 1:N | One encounter class → many encounters |
| `dim_reason` | `fact_encounters` | `reason_key` | 1:N | Conformed — shared across 3 facts |
| `dim_reason` | `fact_medications` | `reason_key` | 1:N | Conformed — shared across 3 facts |
| `dim_reason` | `fact_procedures` | `reason_key` | 1:N | Conformed — shared across 3 facts |
| `dim_condition` | `fact_conditions` | `condition_key` | 1:N | One condition → many diagnosis events |
| `dim_medication` | `fact_medications` | `medication_key` | 1:N | One medication → many prescriptions |
| `dim_observation_type` | `fact_observations` | `observation_type_key` | 1:N | One observation type → many readings |
| `dim_procedure` | `fact_procedures` | `procedure_key` | 1:N | One procedure → many performances |
| `dim_date` | `fact_encounters` | `start_date_key`, `stop_date_key` | 1:N | **Role-playing**: start & end dates |
| `dim_date` | `fact_observations` | `date_key` | 1:N | Single role |
| `dim_date` | `fact_conditions` | `start_date_key`, `stop_date_key` | 1:N | **Role-playing**: diagnosis & resolution |
| `dim_date` | `fact_medications` | `start_date_key`, `stop_date_key` | 1:N | **Role-playing**: start & stop dates |
| `dim_date` | `fact_procedures` | `date_key` | 1:N | Single role |
| `dim_date` | `fact_dq_quarantine_log` | `batch_date_key` | 1:N | Batch date |

#### Fact-to-Fact Relationships (via Degenerate Dimension)

| Parent Fact | Child Fact | Join Key | Cardinality | Notes |
| --- | --- | --- | --- | --- |
| `fact_encounters` | `fact_observations` | `encounter_id` | 1:N | One encounter → many observations; 2.69% have NULL `encounter_id` (standalone) |
| `fact_encounters` | `fact_conditions` | `encounter_id` | 1:N | One encounter → many condition diagnoses |
| `fact_encounters` | `fact_medications` | `encounter_id` | 1:N | One encounter → many prescriptions |
| `fact_encounters` | `fact_procedures` | `encounter_id` | 1:N | One encounter → many procedures |

#### Relationship Summary Diagram (Text ERD)

```
                          dim_encounter_type
                                |
                         fact_encounters ──────── dim_reason (conformed)
                    /    |    |      |    \         |          \
        dim_patient  dim_provider  dim_organization  dim_payer  dim_date (role-playing)
           |  \           |  \         |               |  \        |
           |   \          |   \        |               |   \       |
     fact_obs  fact_cond  |  fact_med  |          fact_proc  fact_pcp
           |     |        |     |      |               |         |
           |     |        |     |      |               |         |
           └─────┴────────┘     └──────┘               └─────────┘
              (all join back to fact_encounters via encounter_id degenerate dimension)

  dim_provider ──(snowflake FK)──→ dim_organization
  dim_date ──(role-playing)──→ fact_encounters (start_date_key, stop_date_key)
  dim_reason (-1 Unknown Member) ──→ fact_encounters, fact_medications, fact_procedures
```

### 1.5 Business Definitions and Grain

#### Key Business Terms

| Term | Definition |
| --- | --- |
| **Encounter** | A single healthcare visit by a patient to a provider at a facility (ambulatory, emergency, inpatient, wellness, or urgent care) |
| **Observation** | A clinical measurement or test result recorded for a patient (vital signs, lab results, COVID tests) |
| **Condition** | A medical diagnosis assigned to a patient during an encounter (SNOMED-CT coded) |
| **Medication** | A pharmaceutical prescription given to a patient, tracked from start to stop date with cost breakdowns |
| **Procedure** | A medical procedure performed on a patient during an encounter |
| **Coverage Period** | A time span during which a patient was insured by a specific payer |
| **Unknown Member (-1)** | A surrogate key value representing 'No recorded reason' in `dim_reason`; applied when the source data has no reason code (common for routine/wellness visits) |
| **Degenerate Dimension** | The `encounter_id` column stored directly in fact tables (not a separate dimension) used to link child facts back to `fact_encounters` |
| **Conformed Dimension** | A dimension shared across multiple fact tables (e.g., `dim_payer`, `dim_reason`, `dim_date`) |
| **Role-Playing Dimension** | A single dimension joined to the same fact table via multiple FKs representing different roles (e.g., `dim_date` joined as `start_date_key` and `stop_date_key`) |
| **Snowflake** | A dimension that has its own FK to another dimension (e.g., `dim_provider` → `dim_organization`) |
| **Surrogate Key** | A system-generated immutable key using `xxhash64(business_key)` — stable across pipeline refreshes |
| **SCD Type 2** | Slowly Changing Dimension Type 2 — preserves all historical versions of a row with `__START_AT` / `__END_AT` validity periods |
| **Bridge Table** | A fact table that resolves many-to-many relationships (e.g., `fact_conditions` bridges patients and conditions for comorbidity analysis) |

#### Grain Statement per Entity

| Entity | Grain (Level of Detail) | Measures / Facts |
| --- | --- | --- |
| `dim_patient` | One row per patient **version** (SCD2) | — (dimension attributes only) |
| `dim_date` | One row per calendar date | — (calendar attributes only) |
| `dim_condition` | One row per unique condition | — (lookup only) |
| `dim_encounter_type` | One row per encounter classification | — (lookup only) |
| `dim_medication` | One row per unique medication | — (lookup only) |
| `dim_observation_type` | One row per observation type | — (lookup only) |
| `dim_organization` | One row per healthcare facility | — (lookup only) |
| `dim_payer` | One row per insurance provider | — (lookup only) |
| `dim_procedure` | One row per unique procedure | — (lookup only) |
| `dim_provider` | One row per physician/provider | — (lookup only) |
| `dim_reason` | One row per unique reason + Unknown Member | — (lookup only) |
| `fact_encounters` | One row per healthcare encounter | `duration_minutes`, `base_encounter_cost`, `total_claim_cost`, `payer_coverage`, `patient_out_of_pocket`, `is_covid_related` |
| `fact_observations` | One row per clinical observation | `value_numeric`, `value_text` |
| `fact_conditions` | One row per condition diagnosis occurrence | `condition_duration_days`, `is_active` |
| `fact_medications` | One row per medication prescription | `dispenses`, `base_cost`, `total_cost`, `payer_coverage`, `patient_out_of_pocket`, `treatment_duration_days`, `is_active` |
| `fact_procedures` | One row per procedure performed | `base_cost` |
| `fact_payer_coverage_period` | One row per patient–payer coverage period | `start_year`, `end_year`, `duration_years` |
| `fact_dq_quarantine_log` | One row per DQ rejection event per batch | `rejected_row_count` |

#### Kimball Design Patterns Applied

| Pattern | Implementation |
| --- | --- |
| **Surrogate Keys** | All dimensions use `xxhash64(business_key)` as PK; all fact PKs use `xxhash64(composite_key)` — except `fact_encounters` which uses the natural UUID `encounter_id` |
| **Unknown Member** | `dim_reason` row with `reason_key = -1` for NULL reasons; applied in `fact_encounters`, `fact_medications`, `fact_procedures` |
| **SCD Type 2** | `dim_patient` via `dlt.apply_changes` with `__START_AT` / `__END_AT` / `is_current` columns |
| **SCD Type 1** | All other dimensions (overwrite on change) |
| **Conformed Dimensions** | `dim_date`, `dim_payer`, `dim_reason` shared across multiple fact tables |
| **Role-Playing Dimensions** | `dim_date` joined via multiple FKs (start/stop dates) |
| **Snowflake** | `dim_provider` → `dim_organization` (FK within a dimension) |
| **Degenerate Dimension** | `encounter_id` stored as fact column, referenced by child facts |
| **Bridge Table** | `fact_conditions` bridges `dim_patient` ↔ `dim_condition` for comorbidity analysis |
| **Factless Fact Table** | `fact_conditions` has no additive measures — records events (diagnosis occurrences) |

## Part 2 — Physical Data Model

### 2.1 Data Dictionary — Dimension Tables

#### 2.1.1 dim\_patient (MATERIALIZED\_VIEW — SCD Type 2)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `patient_key` | BIGINT | YES | **PK** | Surrogate primary key for the patient; immutable even when patient attributes update (SCD2) |
| 1 | `patient_id` | STRING | YES | — | Original patient identifier from the Synthea source system (Business Key) |
| 2 | `first_name` | STRING | YES | — | Patient first name |
| 3 | `last_name` | STRING | YES | — | Patient last name |
| 4 | `birth_date` | DATE | YES | — | Patient date of birth |
| 5 | `death_date` | DATE | YES | — | Date of death, if applicable. NULL indicates the patient is still alive |
| 6 | `gender` | STRING | YES | — | Gender (M/F) |
| 7 | `race` | STRING | YES | — | Race/Ancestry |
| 8 | `ethnicity` | STRING | YES | — | Ethnic origin |
| 9 | `marital_status` | STRING | YES | — | Marital status (Married/Single). Commonly NULL for children |
| 10 | `address` | STRING | YES | — | Detailed street address of the patient |
| 11 | `city` | STRING | YES | — | City of residence |
| 12 | `state` | STRING | YES | — | State (constant: Massachusetts) |
| 13 | `county` | STRING | YES | — | County — finest geographic granularity available for analysis |
| 14 | `zip` | STRING | YES | — | ZIP/Postal code |
| 15 | `lat` | DOUBLE | YES | — | Latitude coordinate of the patient address |
| 16 | `lon` | DOUBLE | YES | — | Longitude coordinate of the patient address |
| 17 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |
| 18 | `__START_AT` | TIMESTAMP | YES | — | SCD2: timestamp from which this version became active |
| 19 | `__END_AT` | TIMESTAMP | YES | — | SCD2: timestamp this version stopped being active (NULL = current version) |
| 20 | `is_current` | BOOLEAN | YES | — | Flag: true if this is the current active patient profile |

#### 2.1.2 dim\_date (MATERIALIZED\_VIEW)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `date_key` | INT | YES | **PK** | Primary key for date in YYYYMMDD numeric format |
| 1 | `full_date` | DATE | YES | — | Full date in DATE format |
| 2 | `day` | INT | YES | — | Day of the month (1-31) |
| 3 | `day_name` | STRING | YES | — | Day name in English (Monday, Tuesday...) |
| 4 | `month` | INT | YES | — | Month number from 1 to 12 |
| 5 | `month_name` | STRING | YES | — | Month name in English (January, February...) |
| 6 | `quarter` | INT | YES | — | Calendar quarter from 1 to 4 |
| 7 | `year` | INT | YES | — | Calendar year |
| 8 | `week_of_year` | INT | YES | — | ISO week number within the year (1-53) |
| 9 | `is_weekend` | BOOLEAN | YES | — | Indicates whether the day is a weekend (Saturday/Sunday) |

#### 2.1.3 dim\_condition (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `condition_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of condition_code) |
| 1 | `condition_code` | STRING | YES | — | Standard medical diagnosis code (SNOMED-CT) |
| 2 | `condition_description` | STRING | YES | — | Descriptive medical name of the condition |
| 3 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.4 dim\_encounter\_type (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `encounter_type_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of encounter_code) |
| 1 | `encounter_code` | STRING | YES | — | Standard code identifying the encounter type/class |
| 2 | `encounter_description` | STRING | YES | — | Detailed description of the visit class |
| 3 | `encounter_class` | STRING | YES | — | Classification category (ambulatory/emergency/inpatient/wellness/urgentcare) |
| 4 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.5 dim\_medication (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `medication_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of medication_code) |
| 1 | `medication_code` | STRING | YES | — | Standard medication code |
| 2 | `medication_description` | STRING | YES | — | Brand or generic name of the medication |
| 3 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.6 dim\_observation\_type (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `observation_type_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of observation_code) |
| 1 | `observation_code` | STRING | YES | — | Standard observation code |
| 2 | `observation_description` | STRING | YES | — | Observation name (e.g., Body Height, Body Weight, Blood Pressure) |
| 3 | `units` | STRING | YES | — | Unit of measurement for the observation (e.g., cm, kg, mmHg) |
| 4 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.7 dim\_organization (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `organization_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of organization_id) |
| 1 | `organization_id` | STRING | YES | — | Original organization identifier from Synthea (Business Key) |
| 2 | `organization_name` | STRING | YES | — | Name of the hospital or medical facility |
| 3 | `address` | STRING | YES | — | Street address of the facility |
| 4 | `city` | STRING | YES | — | City where the facility is located |
| 5 | `state` | STRING | YES | — | State (constant: Massachusetts) |
| 6 | `zip` | STRING | YES | — | ZIP/Postal code of the facility |
| 7 | `lat` | DOUBLE | YES | — | Latitude coordinate of the facility |
| 8 | `lon` | DOUBLE | YES | — | Longitude coordinate of the facility |
| 9 | `phone` | STRING | YES | — | Phone number of the facility |
| 10 | `revenue` | DECIMAL | YES | — | Facility revenue |
| 11 | `utilization` | BIGINT | YES | — | Facility utilization metric |
| 12 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.8 dim\_payer (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `payer_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of payer_id) |
| 1 | `payer_id` | STRING | YES | — | Original insurance payer identifier from Synthea (Business Key) |
| 2 | `payer_name` | STRING | YES | — | Name of the insurance company |
| 3 | `address` | STRING | YES | — | Payer address |
| 4 | `city` | STRING | YES | — | Payer city |
| 5 | `state_headquartered` | STRING | YES | — | State where the payer is headquartered |
| 6 | `zip` | STRING | YES | — | Payer ZIP code |
| 7 | `phone` | STRING | YES | — | Payer phone number |
| 8 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.9 dim\_procedure (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `procedure_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of procedure_code) |
| 1 | `procedure_code` | STRING | YES | — | Standard medical procedure code |
| 2 | `procedure_description` | STRING | YES | — | Description of the medical procedure |
| 3 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.10 dim\_provider (STREAMING\_TABLE — Snowflake)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `provider_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of provider_id) |
| 1 | `provider_id` | STRING | YES | — | Original provider identifier from Synthea (Business Key) |
| 2 | `provider_name` | STRING | YES | — | Name of the physician or healthcare provider |
| 3 | `gender` | STRING | YES | — | Gender of the physician/provider (M/F) |
| 4 | `speciality` | STRING | YES | — | Medical specialty |
| 5 | `organization_key` | BIGINT | YES | **FK** → `dim_organization.organization_key` | Identifies the hospital/facility this provider is employed at |
| 6 | `address` | STRING | YES | — | Provider address |
| 7 | `city` | STRING | YES | — | Provider city |
| 8 | `state` | STRING | YES | — | Provider state |
| 9 | `zip` | STRING | YES | — | Provider ZIP code |
| 10 | `lat` | DOUBLE | YES | — | Latitude coordinate |
| 11 | `lon` | DOUBLE | YES | — | Longitude coordinate |
| 12 | `utilization` | BIGINT | YES | — | Provider utilization metric |
| 13 | `_ingested_at` | TIMESTAMP | YES | — | Pipeline ingestion timestamp |

#### 2.1.11 dim\_reason (MATERIALIZED\_VIEW — Conformed)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `reason_key` | BIGINT | YES | **PK** | Primary key; value -1 represents 'No recorded reason' (Unknown Member) |
| 1 | `reason_code` | STRING | YES | — | Standard medical code for the visit/medication/procedure reason (SNOMED-CT). NULL for Unknown Member row |
| 2 | `reason_description` | STRING | YES | — | Description of the medical reason (diagnosis associated with the visit/medication) |

### 2.2 Data Dictionary — Fact Tables

#### 2.2.1 fact\_encounters (STREAMING\_TABLE — Hub Fact)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `encounter_id` | STRING | YES | **PK** | Primary key (UUID Business Key). Also a Degenerate Dimension referenced by other fact tables |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the patient of this encounter |
| 2 | `provider_key` | BIGINT | YES | **FK** → `dim_provider.provider_key` | Identifies the physician/provider who conducted this encounter |
| 3 | `organization_key` | BIGINT | YES | **FK** → `dim_organization.organization_key` | Identifies the hospital/facility where this encounter took place |
| 4 | `payer_key` | BIGINT | YES | **FK** → `dim_payer.payer_key` | Identifies the insurance provider covering this encounter |
| 5 | `encounter_type_key` | BIGINT | YES | **FK** → `dim_encounter_type.encounter_type_key` | Identifies the classification of this visit |
| 6 | `reason_key` | BIGINT | YES | **FK** → `dim_reason.reason_key` | Identifies the medical reason for this visit (-1 = Unknown Member) |
| 7 | `start_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the encounter started (role-playing) |
| 8 | `stop_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the encounter ended (role-playing) |
| 9 | `start_datetime` | TIMESTAMP | YES | — | Exact timestamp the encounter started |
| 10 | `stop_datetime` | TIMESTAMP | YES | — | Exact timestamp the encounter ended |
| 11 | `duration_minutes` | INT | YES | — | Encounter duration in minutes (stop - start) |
| 12 | `base_encounter_cost` | DECIMAL(10,2) | YES | — | Base cost of the encounter prior to insurance coverage calculation |
| 13 | `total_claim_cost` | DECIMAL(10,2) | YES | — | Total insurance claim cost billed |
| 14 | `payer_coverage` | DECIMAL(10,2) | YES | — | Amount actually covered by the insurance provider |
| 15 | `patient_out_of_pocket` | DECIMAL(11,2) | YES | — | Amount paid out-of-pocket = total\_claim\_cost - payer\_coverage |
| 16 | `is_covid_related` | BOOLEAN | YES | — | Flag indicating if the visit is COVID-19 related |

#### 2.2.2 fact\_observations (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `observation_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of patient + encounter + code + timestamp) |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the patient this observation was recorded for |
| 2 | `encounter_id` | STRING | YES | **DD** → `fact_encounters.encounter_id` | Degenerate dimension — nullable (2.69% are standalone) |
| 3 | `observation_type_key` | BIGINT | YES | **FK** → `dim_observation_type.observation_type_key` | Identifies the type of measurement/test taken |
| 4 | `date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the observation was recorded |
| 5 | `observation_datetime` | TIMESTAMP | YES | — | Exact timestamp of the observation (for trend analysis ordering) |
| 6 | `value_numeric` | DOUBLE | YES | — | Numeric observation value (e.g., 98.6 for temperature); NULL if textual |
| 7 | `value_text` | STRING | YES | — | Textual observation value (e.g., Positive/Negative); NULL if numeric |

#### 2.2.3 fact\_conditions (STREAMING\_TABLE — Factless / Bridge)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `condition_fact_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of patient + encounter + condition + start date) |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the diagnosed patient (also bridge key for comorbidity) |
| 2 | `encounter_id` | STRING | YES | **DD** → `fact_encounters.encounter_id` | Degenerate dimension linking to the encounter |
| 3 | `condition_key` | BIGINT | YES | **FK** → `dim_condition.condition_key` | Identifies the diagnosed condition |
| 4 | `start_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the condition was first diagnosed (role-playing) |
| 5 | `stop_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the condition was resolved (role-playing, NULL = active) |
| 6 | `condition_duration_days` | INT | YES | — | Duration of the condition in days; NULL for ongoing conditions |
| 7 | `is_active` | BOOLEAN | YES | — | Flag: true if condition is ongoing (stop\_date IS NULL) |

#### 2.2.4 fact\_medications (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `medication_fact_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of patient + encounter + medication + start date) |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the patient this medication was prescribed to |
| 2 | `encounter_id` | STRING | YES | **DD** → `fact_encounters.encounter_id` | Degenerate dimension linking to the prescribing encounter |
| 3 | `payer_key` | BIGINT | YES | **FK** → `dim_payer.payer_key` | Identifies the insurance provider covering this medication |
| 4 | `medication_key` | BIGINT | YES | **FK** → `dim_medication.medication_key` | Identifies the prescribed medication |
| 5 | `reason_key` | BIGINT | YES | **FK** → `dim_reason.reason_key` | Identifies the medical reason for this prescription (-1 = Unknown) |
| 6 | `start_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the medication was first prescribed (role-playing) |
| 7 | `stop_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the medication was stopped (role-playing, NULL = active) |
| 8 | `dispenses` | INT | YES | — | Total number of times the medication was dispensed |
| 9 | `base_cost` | DECIMAL(10,2) | YES | — | Base cost of the medication before insurance coverage |
| 10 | `total_cost` | DECIMAL(10,2) | YES | — | Total billed cost for this medication across all dispenses |
| 11 | `payer_coverage` | DECIMAL(10,2) | YES | — | Amount actually covered by the insurance provider |
| 12 | `patient_out_of_pocket` | DECIMAL(11,2) | YES | — | Amount paid out-of-pocket = total\_cost - payer\_coverage |
| 13 | `treatment_duration_days` | INT | YES | — | Treatment duration in days; NULL for active prescriptions |
| 14 | `is_active` | BOOLEAN | YES | — | Flag: true if patient is currently taking the medication |

#### 2.2.5 fact\_procedures (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `procedure_fact_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of patient + encounter + procedure + date) |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the patient this procedure was performed on |
| 2 | `encounter_id` | STRING | YES | **DD** → `fact_encounters.encounter_id` | Degenerate dimension linking to the encounter |
| 3 | `procedure_key` | BIGINT | YES | **FK** → `dim_procedure.procedure_key` | Identifies the medical procedure performed |
| 4 | `reason_key` | BIGINT | YES | **FK** → `dim_reason.reason_key` | Identifies the medical reason for this procedure (-1 = Unknown) |
| 5 | `date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date the procedure was performed |
| 6 | `base_cost` | DECIMAL(10,2) | YES | — | Base cost of the procedure (no payer breakdown in source) |

#### 2.2.6 fact\_payer\_coverage\_period (STREAMING\_TABLE)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `coverage_fact_key` | BIGINT | YES | **PK** | Surrogate primary key (xxhash64 of patient + payer + start\_year) |
| 1 | `patient_key` | BIGINT | YES | **FK** → `dim_patient.patient_key` | Identifies the covered patient |
| 2 | `payer_key` | BIGINT | YES | **FK** → `dim_payer.payer_key` | Identifies the insurance provider for this coverage period |
| 3 | `start_year` | INT | YES | — | Coverage start year |
| 4 | `end_year` | INT | YES | — | Coverage end year (always has a value in the source data) |
| 5 | `ownership` | STRING | YES | — | Relationship type (Self/Spouse/Guardian) |
| 6 | `duration_years` | INT | YES | — | Calculated: end\_year - start\_year |

#### 2.2.7 fact\_dq\_quarantine\_log (STREAMING\_TABLE — Governance)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `quarantine_key` | BIGINT | YES | **PK** | Surrogate primary key of this log table |
| 1 | `source_table` | STRING | YES | — | Source table name where the rejection occurred |
| 2 | `rule_violated` | STRING | YES | — | Description of the violated quality rule (e.g., STOP < START) |
| 3 | `rejected_row_count` | BIGINT | YES | — | Number of rejected records in this batch |
| 4 | `batch_date_key` | INT | YES | **FK** → `dim_date.date_key` | Calendar date of the batch run |
| 5 | `logged_at` | TIMESTAMP | YES | — | Exact timestamp this quarantine metric was computed |

#### 2.2.8 star\_schema\_metadata (MANAGED — Constraint Catalog)

| # | Column | Data Type | Nullable | Key | Description |
| --- | --- | --- | --- | --- | --- |
| 0 | `table_name` | STRING | YES | — | Fully qualified Gold table name |
| 1 | `column_name` | STRING | YES | — | The column acting as PK or FK |
| 2 | `constraint_type` | STRING | YES | — | PRIMARY\_KEY or FOREIGN\_KEY |
| 3 | `constraint_name` | STRING | YES | — | Named constraint identifier (e.g., pk\_dim\_patient, fk\_fe\_patient) |
| 4 | `references_table` | STRING | YES | — | For FK: the dimension table this FK references. NULL for PK |
| 5 | `references_column` | STRING | YES | — | For FK: the PK column in the referenced dimension. NULL for PK |
| 6 | `role` | STRING | YES | — | dimension, fact, or bridge — Kimball role of the table |

> **Note:** `PK` = Primary Key, `FK` = Foreign Key, `DD` = Degenerate Dimension (direct join key to `fact_encounters`)

### 2.3 Physical Delta Lake Configurations

All Gold tables are managed by a single Lakeflow Spark Declarative Pipeline (SDP) and materialized as Delta Lake tables in Unity Catalog. The following physical configurations apply uniformly unless noted otherwise.

#### 2.3.1 Delta Lake Feature Matrix (All Gold Tables)

| Delta Feature | Property | Value | Purpose |
| --- | --- | --- | --- |
| **Change Data Feed (CDF)** | `delta.enableChangeDataFeed` | `true` | Enables incremental CDC reads for downstream consumers |
| **Deletion Vectors** | `delta.enableDeletionVectors` | `true` | Improves UPDATE/MERGE/DELETE performance by tracking soft deletes |
| **Row Tracking** | `delta.enableRowTracking` | `true` | Assigns stable row IDs and commit versions for lineage tracking |
| **ZSTD Compression** | `delta.parquet.compression.codec` | `zstd` | Higher compression ratio than Snappy, reduces storage footprint |
| **Min Reader Version** | `delta.minReaderVersion` | `3` | Enables DV and CDF reads |
| **Min Writer Version** | `delta.minWriterVersion` | `7` | Enables clustering, DV, row tracking, v2 checkpoints |
| **V2 Checkpoint** | `delta.checkpointPolicy` | `v2` | Faster, more efficient checkpointing (facts with clustering) |
| **Predictive Optimization** | `predictiveOptimization` | `ENABLE` | Auto-runs OPTIMIZE, VACUUM on all gold tables |

#### 2.3.2 Partitioning Strategy

**No partitioning is used on any Gold table.** All tables rely on **Liquid Clustering** (for the 3 largest fact tables) or default file-level optimization via Predictive Optimization. This follows Databricks best practice: Liquid Clustering replaces partitioning for most use cases.

#### 2.3.3 Liquid Clustering Configuration

Liquid Clustering is applied to the 3 largest fact tables, using the most common filter columns from dashboard and analytical queries:

| Table | Clustering Keys | Benchmark Speedup | Rows | Rationale |
| --- | --- | --- | --- | --- |
| `fact_observations` | `CLUSTER BY (patient_key, date_key)` | **47.2%** | 16,219,969 | Patient-level trend analysis + date-range queries are the dominant access patterns |
| `fact_medications` | `CLUSTER BY (patient_key, start_date_key)` | **28.4%** | 4,226,915 | Patient medication history + prescription date filtering |
| `fact_encounters` | `CLUSTER BY (patient_key, start_date_key)` | **17.5%** | 3,183,531 | Patient encounter history + date-range analysis |

> **Note:** `fact_conditions`, `fact_procedures`, `fact_payer_coverage_period`, `fact_dq_quarantine_log`, and all dimension tables do NOT have Liquid Clustering. They rely on Predictive Optimization for file compaction. Their smaller size (<1.2M rows) makes clustering unnecessary.

#### 2.3.4 SCD Type 2 Configuration (dim\_patient)

| Property | Value |
| --- | --- |
| **Implementation** | `dlt.apply_changes()` in the SDP pipeline |
| **Backing Table** | `dim_patient_scd` (STREAMING\_TABLE) — internal storage |
| **Exposed View** | `dim_patient` (MATERIALIZED\_VIEW) — adds `is_current` derived column |
| **Sequence Key** | `_ingested_at` (TIMESTAMP) — pipeline ingestion timestamp determines version order |
| **SCD2 Columns** | `__START_AT` (TIMESTAMP) — version start; `__END_AT` (TIMESTAMP) — version end (NULL = current) |
| **Derived Column** | `is_current` (BOOLEAN) — computed as `__END_AT IS NULL`; use `WHERE is_current = true` for latest patient profile |
| **Apply Mode** | SCD Type 2 — preserves all historical versions; multiple rows per patient when attributes change |

#### 2.3.5 Table Type & Materialization

| Table | Type | Pipeline Materialization | Refresh |
| --- | --- | --- | --- |
| `dim_patient` | MATERIALIZED\_VIEW | Derived from `dim_patient_scd` via SDP | MANUAL (triggered on pipeline update) |
| `dim_date` | MATERIALIZED\_VIEW | Calendar generation logic in SDP | MANUAL |
| `dim_reason` | MATERIALIZED\_VIEW | Union of reasons from 3 source tables + Unknown Member row | MANUAL |
| `dim_condition` | STREAMING\_TABLE | DLT streaming table with Auto Loader | MANUAL |
| `dim_encounter_type` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_medication` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_observation_type` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_organization` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_payer` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_procedure` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_provider` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `dim_patient_scd` | STREAMING\_TABLE | Internal SCD2 backing via `dlt.apply_changes` | MANUAL |
| `fact_encounters` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_observations` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_conditions` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_medications` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_procedures` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_payer_coverage_period` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `fact_dq_quarantine_log` | STREAMING\_TABLE | DLT streaming table | MANUAL |
| `star_schema_metadata` | MANAGED | Manually populated constraint catalog | N/A |

#### 2.3.6 Primary & Foreign Key Catalog

The following constraints are documented in the `star_schema_metadata` table (queryable catalog). DLT-managed tables do not support `ALTER TABLE ADD CONSTRAINT`, so this metadata table serves as the system of record.

**Primary Keys:**

| Table | PK Column | Constraint Name | Data Type |
| --- | --- | --- | --- |
| `dim_patient` | `patient_key` | `pk_dim_patient` | BIGINT |
| `dim_patient_scd` | `patient_key` | `pk_dim_patient_scd` | BIGINT |
| `dim_date` | `date_key` | `pk_dim_date` | INT |
| `dim_condition` | `condition_key` | `pk_dim_condition` | BIGINT |
| `dim_encounter_type` | `encounter_type_key` | `pk_dim_encounter_type` | BIGINT |
| `dim_medication` | `medication_key` | `pk_dim_medication` | BIGINT |
| `dim_observation_type` | `observation_type_key` | `pk_dim_observation_type` | BIGINT |
| `dim_organization` | `organization_key` | `pk_dim_organization` | BIGINT |
| `dim_payer` | `payer_key` | `pk_dim_payer` | BIGINT |
| `dim_procedure` | `procedure_key` | `pk_dim_procedure` | BIGINT |
| `dim_provider` | `provider_key` | `pk_dim_provider` | BIGINT |
| `dim_reason` | `reason_key` | `pk_dim_reason` | BIGINT |
| `fact_encounters` | `encounter_id` | `pk_fact_encounters` | STRING |
| `fact_observations` | `observation_key` | `pk_fact_observations` | BIGINT |
| `fact_conditions` | `condition_fact_key` | `pk_fact_conditions` | BIGINT |
| `fact_medications` | `medication_fact_key` | `pk_fact_medications` | BIGINT |
| `fact_procedures` | `procedure_fact_key` | `pk_fact_procedures` | BIGINT |
| `fact_payer_coverage_period` | `coverage_fact_key` | `pk_fact_payer_coverage_period` | BIGINT |
| `fact_dq_quarantine_log` | `quarantine_key` | `pk_fact_dq_quarantine_log` | BIGINT |

**Foreign Keys (27 total):**

| Fact Table | FK Column | Constraint Name | References |
| --- | --- | --- | --- |
| `dim_provider` | `organization_key` | `fk_dp_organization` | `dim_organization.organization_key` |
| `fact_encounters` | `patient_key` | `fk_fe_patient` | `dim_patient.patient_key` |
| `fact_encounters` | `provider_key` | `fk_fe_provider` | `dim_provider.provider_key` |
| `fact_encounters` | `organization_key` | `fk_fe_organization` | `dim_organization.organization_key` |
| `fact_encounters` | `payer_key` | `fk_fe_payer` | `dim_payer.payer_key` |
| `fact_encounters` | `encounter_type_key` | `fk_fe_encounter_type` | `dim_encounter_type.encounter_type_key` |
| `fact_encounters` | `reason_key` | `fk_fe_reason` | `dim_reason.reason_key` |
| `fact_encounters` | `start_date_key` | `fk_fe_start_date` | `dim_date.date_key` |
| `fact_encounters` | `stop_date_key` | `fk_fe_stop_date` | `dim_date.date_key` |
| `fact_observations` | `patient_key` | `fk_fo_patient` | `dim_patient.patient_key` |
| `fact_observations` | `observation_type_key` | `fk_fo_observation_type` | `dim_observation_type.observation_type_key` |
| `fact_observations` | `date_key` | `fk_fo_date` | `dim_date.date_key` |
| `fact_conditions` | `patient_key` | `fk_fc_patient` | `dim_patient.patient_key` |
| `fact_conditions` | `condition_key` | `fk_fc_condition` | `dim_condition.condition_key` |
| `fact_conditions` | `start_date_key` | `fk_fc_start_date` | `dim_date.date_key` |
| `fact_conditions` | `stop_date_key` | `fk_fc_stop_date` | `dim_date.date_key` |
| `fact_medications` | `patient_key` | `fk_fm_patient` | `dim_patient.patient_key` |
| `fact_medications` | `payer_key` | `fk_fm_payer` | `dim_payer.payer_key` |
| `fact_medications` | `medication_key` | `fk_fm_medication` | `dim_medication.medication_key` |
| `fact_medications` | `reason_key` | `fk_fm_reason` | `dim_reason.reason_key` |
| `fact_medications` | `start_date_key` | `fk_fm_start_date` | `dim_date.date_key` |
| `fact_medications` | `stop_date_key` | `fk_fm_stop_date` | `dim_date.date_key` |
| `fact_procedures` | `patient_key` | `fk_fp_patient` | `dim_patient.patient_key` |
| `fact_procedures` | `procedure_key` | `fk_fp_procedure` | `dim_procedure.procedure_key` |
| `fact_procedures` | `reason_key` | `fk_fp_reason` | `dim_reason.reason_key` |
| `fact_procedures` | `date_key` | `fk_fp_date` | `dim_date.date_key` |
| `fact_payer_coverage_period` | `patient_key` | `fk_fpcp_patient` | `dim_patient.patient_key` |
| `fact_payer_coverage_period` | `payer_key` | `fk_fpcp_payer` | `dim_payer.payer_key` |
| `fact_dq_quarantine_log` | `batch_date_key` | `fk_fdq_batch_date` | `dim_date.date_key` |

---

### 2.4 Storage & Infrastructure

| Property | Value |
| --- | --- |
| **Storage** | AWS S3 (Databricks-managed UC storage) |
| **Catalog** | `covid19_lakehouse` (Unity Catalog) |
| **Schema** | `gold` |
| **Table Owner** | `ahmednabil332002@gmail.com` |
| **Pipeline** | Lakeflow Spark Declarative Pipeline (SDP) — Pipeline ID: `2c75e768-b847-40c6-a72f-a86dca2a5272` |
| **Refresh** | MANUAL (triggered on pipeline update) |
| **Total Entities** | 18 core tables + 5 metric views + 1 metadata table + 1 internal SCD2 table = 25 objects |
| **Total Rows (facts)** | \~26.2M across 7 fact tables |
| **Total Rows (dims)** | \~246K across 11 dimension tables |
| **Semantic Layer** | 5 Metric Views (fact\_encounters, fact\_conditions, fact\_medications, fact\_observations, fact\_procedures) |